# Paired L9 check after U3

This checks whether the first forget request makes later learning of task 9 more damaging to retained task 0. One arm learns L9 from the E08 checkpoint without U3. The other learns L9 from the saved post-U3 checkpoint. Both use the same task data, seed, and learning settings. Compare the change in task 0 accuracy during L9, not just its final accuracy.

This is a one-seed diagnostic using the older all-images training split. Use a fresh Colab GPU runtime. Each arm saves its own checkpoint and JSON logs. The comparison is saved before Drive is flushed and the GPU is released.


In [ ]:
from urllib.request import urlretrieve
import runpy

urlretrieve(
    "https://raw.githubusercontent.com/sumitasthana/CARK/main/scripts/colab_setup.py",
    "/content/uncle_colab_setup.py",
)
_ = runpy.run_path("/content/uncle_colab_setup.py", run_name="__main__")


## Check both starting points

The E08 checkpoint is before U3. The second checkpoint is immediately after U3 in the completed sequence. This cell checks their request histories, accuracies, and settings before any GPU training.


In [ ]:
from dataclasses import replace
from pathlib import Path
from uncle.config import Config
from uncle.notebook import inspect_checkpoint, prepare_session

ROOT = Path("/content/drive/MyDrive/uncle/E08_forgetting_trace")
E08 = ROOT / "before_forgetting.pt"
POST_U3 = ROOT / "full_sequence_27step_candidate/checkpoint_seq1_resnet50_seed0_after_request02_forget3.pt"
DATA = Path("/content/data/tiny-imagenet-200")
OUTPUT = ROOT / "paired_L9_probe"

e08 = prepare_session(
    E08, expected_history=[("learn", "3"), ("learn", "0")],
    expected_accuracies={"3": 26.0, "0": 44.6},
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
post = inspect_checkpoint(
    POST_U3, expected_history=[("learn", "3"), ("learn", "0"), ("forget", "3")],
    expected_accuracies={"3": 12.6, "0": 41.0},
    expected_config={"dataset": "tiny_imagenet", "backbone": "resnet50"},
)
assert e08["forgotten"] == [] and post["forgotten"] == ["3"]
full_config = Config(**post["config"])
assert full_config.requests[:4] == (("learn", "3"), ("learn", "0"),
                                    ("forget", "3"), ("learn", "9"))
assert (full_config.gamma, full_config.forgetting_learning_rate,
        full_config.burn_in, full_config.burn_in_decay, full_config.burn_in_min) == (5e-6, 1e-5, 27, 1.0, 27)
assert e08["config"]["seed"] == full_config.seed
assert e08["config"]["tasks"] == post["config"]["tasks"]
without_u3 = replace(full_config, requests=(("learn", "3"), ("learn", "0"), ("learn", "9")))
after_u3 = replace(full_config, requests=full_config.requests[:4])
print("Without U3:", E08, "then L9")
print("After U3:", POST_U3, "then L9")
print("Output:", OUTPUT)


## Run L9 once from each checkpoint

Each arm has a separate resumable checkpoint. If Colab stops between arms, run these cells again in a fresh GPU session. Completed arms resume without repeating L9.


In [ ]:
from uncle.experiments import run_experiment

arms = {
    "without_u3": (E08, without_u3),
    "after_u3": (POST_U3, after_u3),
}
runs = {}
for name, (source, config) in arms.items():
    print("Running", name, flush=True)
    runs[name] = run_experiment(
        sequence=1, config=config, starting_checkpoint=source,
        root=DATA, download=True, output=OUTPUT / name,
        checkpoint=True, resume=True, verbose=True, progress=True,
    )


## Save the comparison

Positive task 0 loss means L9 reduced task 0 accuracy. The difference between losses is the extra loss in the after-U3 arm. The original full-sequence L9 result is shown as a replay check.


In [ ]:
import json
import os

reference_path = ROOT / "full_sequence_27step_candidate/history_seq1_resnet50_seed0.json"
reference = json.loads(reference_path.read_text(encoding="utf-8"))[3]
assert (reference["action"], reference["task"]) == ("learn", "9")
comparison = {
    "status": "complete",
    "question": "Does U3 increase task 0 accuracy loss during L9?",
    "arms": {},
    "original_full_run_l9": {
        "task0_before": reference["before"]["0"],
        "task0_after": reference["after"]["0"],
        "task9_after": reference["after"]["9"],
    },
}
for name, (source, config) in arms.items():
    result = runs[name]
    assert len(result["history"]) == len(config.requests)
    row = result["history"][-1]
    assert (row["action"], row["task"]) == ("learn", "9")
    stem = f"seq1_{config.backbone}_seed{config.seed}"
    origin = json.loads((OUTPUT / name / f"origin_{stem}.json").read_text())
    comparison["arms"][name] = {
        "source_checkpoint": str(source),
        "source_sha256": origin["starting_checkpoint_sha256"],
        "task0_before": row["before"]["0"],
        "task0_after": row["after"]["0"],
        "task0_loss_points": row["before"]["0"] - row["after"]["0"],
        "task9_after": row["after"]["9"],
        "task3_before": row["before"]["3"],
        "task3_after": row["after"]["3"],
        "git_commit": result["environment"]["git_commit"],
    }
comparison["extra_task0_loss_after_u3"] = (
    comparison["arms"]["after_u3"]["task0_loss_points"]
    - comparison["arms"]["without_u3"]["task0_loss_points"]
)
OUTPUT.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT / "paired_L9_comparison.json"
temporary = report_path.with_suffix(".json.writing")
temporary.write_text(json.dumps(comparison, indent=2) + "\n", encoding="utf-8")
os.replace(temporary, report_path)
print(json.dumps(comparison, indent=2))
print("Saved:", report_path)


## Release the GPU after saving

This verifies both arms and the comparison, flushes Drive writes, then disconnects the GPU runtime. If a check fails, it stays connected. Open [the CPU review notebook](08_review_paired_L9_probe.ipynb) after disconnecting.


In [ ]:
from google.colab import drive, runtime

assert json.loads(report_path.read_text(encoding="utf-8")) == comparison
for name, (_, config) in arms.items():
    stem = f"seq1_{config.backbone}_seed{config.seed}"
    folder = OUTPUT / name
    for filename in (f"checkpoint_{stem}.pt", f"origin_{stem}.json",
                     f"environment_{stem}.json", f"history_{stem}.json",
                     f"costs_{stem}.json", f"summary_{stem}.json"):
        path = folder / filename
        assert path.is_file() and path.stat().st_size > 0, path
    saved_history = json.loads((folder / f"history_{stem}.json").read_text())
    assert len(saved_history) == len(config.requests)
    assert (saved_history[-1]["action"], saved_history[-1]["task"]) == ("learn", "9")
print("Both L9 runs and comparison saved. Flushing Drive writes.", flush=True)
drive.flush_and_unmount(timeout_ms=600000)
print("Drive flushed. Releasing GPU runtime.", flush=True)
runtime.unassign()
